1. Setup and load

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # lets the notebook import from src/

import pandas as pd
from src.data_loader import load_all

pd.set_option("display.max_columns", 50)
dfs = load_all()

2. Overview of all tables

In [3]:
overview = pd.DataFrame({
    name: {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
        "duplicate_rows": df.duplicated().sum(),
    }
    for name, df in dfs.items()
}).T
overview

,rows,columns,memory_mb,duplicate_rows
customers,99441.0,5.0,27.9,0.0
geolocation,1000163.0,5.0,135.7,261831.0
order_items,112650.0,7.0,31.0,0.0
payments,103886.0,5.0,17.0,0.0
reviews,99224.0,7.0,29.1,0.0
orders,99441.0,8.0,25.9,0.0
products,32951.0,9.0,6.6,0.0
sellers,3095.0,4.0,0.6,0.0
category_translation,71.0,2.0,0.0,0.0


3. Missing values per table

In [4]:
for name, df in dfs.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print(f"\n=== {name} ===")
        print(pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(2)}))


=== reviews ===
                        missing    pct
review_comment_title      87656  88.34
review_comment_message    58247  58.70

=== orders ===
                               missing   pct
order_approved_at                  160  0.16
order_delivered_carrier_date      1783  1.79
order_delivered_customer_date     2965  2.98

=== products ===
                            missing   pct
product_category_name           610  1.85
product_name_lenght             610  1.85
product_description_lenght      610  1.85
product_photos_qty              610  1.85
product_weight_g                  2  0.01
product_length_cm                 2  0.01
product_height_cm                 2  0.01
product_width_cm                  2  0.01


4. Observing the core table

In [5]:
orders = dfs["orders"]
display(orders.head())
orders.info()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [6]:
print(orders["order_status"].value_counts())
print("\nPurchase date range:",
      orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Purchase date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


5. Key and integrity checks

In [7]:
customers = dfs["customers"]
items = dfs["order_items"]

print("order_id unique in orders:", orders["order_id"].is_unique)
print("customer_id unique in customers:", customers["customer_id"].is_unique)
print("unique customers (customer_unique_id):", customers["customer_unique_id"].nunique(),
      "of", len(customers), "rows")
print("order_items order_ids missing from orders:",
      (~items["order_id"].isin(orders["order_id"])).sum())
print("orders with no items:",
      (~orders["order_id"].isin(items["order_id"])).sum())

order_id unique in orders: True
customer_id unique in customers: True
unique customers (customer_unique_id): 96096 of 99441 rows
order_items order_ids missing from orders: 0
orders with no items: 775


6. Summary stats for the numeric columns that matter

In [8]:
dfs["order_items"][["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000
